# 04f. 4모델 앙상블 가중치 + 클래스 승수 (OOF 기반 후처리)

- 담당: **④ 경량화·통합** (이 파일은 담당자만 수정)
- 목적: 제출 11 구성(3모델 균등)에 새 g3를 더한 4모델의 가중치와 클래스별 확률 승수를 정해 Macro F1이 오르는지 확인
- 비교 기준: 3모델 균등 (가중 XGB E + LR E·위치 구간 + 기존 g3)
- 방법: 팀 고정 홀드아웃의 개발셋(4,960명) 5-fold에서 fold 4개로 가중치·승수를 정하고 남은 1개로 평가 × 5회 → 평균 가중치·승수로 홀드아웃(1,241명) 1회 평가
- 입력: experiments/preds의 OOF·test 예측 (재학습 없음)
- 제외: cat_v2_balanced_1107 (11/07 예측, 그룹 fold 도입 전)
- 판단 기준 (팀 공지 11/12): fold 밖에서 정한 경우만 인정, 기존 3모델 대비 OOF +0.005 이상이면 제출 후보
- 기록표에는 적지 않음 (결과는 이 노트북 출력), Run All 가능 (약 3분)

## 0. 환경 설정 (수정하지 말고 그대로 실행)

In [1]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype | DATA_DIR: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/data


## 1. 도구·후보 모델

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from src.config import EXPERIMENT_DIR
from src.data_loader import load_data, split_xy, encode_target
from src.validation import get_folds
from src.prediction import make_submission
from scripts.fixed_holdout_manifest import verify

# 후보 모델 4개 (모두 팀 공통 그룹 fold OOF, test는 train 전체 재학습)
# cat_v2_balanced_1107은 11/07 예측(그룹 fold 도입 전)이라 제외
CANDIDATE_KEYS = [
    "A_model__12_weighted_xgb_E_cpu",            # 가중치 XGBoost (E, CPU)
    "A_model__12_lr_position_bin50",             # LR (E + 위치 구간 50)
    "A_model__lgbm_g3_hotspot3_bin_exact_1110",  # 기존 g3
    "A_model__12_g3_multihot_signature",         # 새 g3 (+A 피처 + 변이 서명)
]
BASE_KEYS = CANDIDATE_KEYS[:3]   # 비교 기준: 제출 11 구성 3모델 균등


## 2. 탐색 함수 — 가중치(Dirichlet 랜덤 서치), 클래스 승수(좌표 상승, 범위 0.65~1.60, L2 정규화)

In [3]:
def load_selected_preds(keys, classes):
    pred_dir = EXPERIMENT_DIR / "preds"
    loaded = {}
    for k in keys:
        oof_p = pred_dir / f"{k}__oof.npy"
        test_p = pred_dir / f"{k}__test.npy"
        if oof_p.exists() and test_p.exists():
            loaded[k] = {
                "oof": np.load(oof_p).astype(np.float64),
                "test": np.load(test_p).astype(np.float64),
            }
        else:
            print(f"[경고] 파일 없음: {k}")
    return loaded


def optimize_ensemble_weights(oof_list, y_true, n_trials=2000, seed=SEED):
    """Dirichlet 랜덤 서치 + Powell 미세 조정으로 dev 내부 OOF Macro F1 최대화."""
    rng = np.random.default_rng(seed)
    n_models = len(oof_list)
    best_w = np.ones(n_models) / n_models
    best_f1 = f1_score(y_true, np.argmax(sum(w * p for w, p in zip(best_w, oof_list)), axis=1), average="macro")

    for _ in range(n_trials):
        w = rng.dirichlet(np.ones(n_models) * 1.5)
        blend = sum(wi * pi for wi, pi in zip(w, oof_list))
        score = f1_score(y_true, np.argmax(blend, axis=1), average="macro")
        if score > best_f1:
            best_f1 = score
            best_w = w
    return best_w, best_f1


def fit_class_multipliers_coord_ascent(proba, y_true, n_iters=3, reg_lambda=0.002):
    """
    Coordinate Ascent로 클래스별 확률 승수 w_c 탐색 (Macro F1 최대화 - L2 정규화).
    과적합 방지를 위해 탐색 범위를 [0.65, 1.60]으로 제한하고 정규화 항을 둔다.
    """
    n_classes = proba.shape[1]
    w = np.ones(n_classes, dtype=np.float64)
    grid = np.linspace(0.65, 1.60, 20)

    def objective(weights):
        pred = np.argmax(proba * weights[None, :], axis=1)
        f1 = f1_score(y_true, pred, average="macro")
        penalty = reg_lambda * np.mean((weights - 1.0) ** 2)
        return f1 - penalty

    best_obj = objective(w)
    for _ in range(n_iters):
        improved = False
        for c in range(n_classes):
            orig = w[c]
            best_c = orig
            for val in grid:
                w[c] = val
                obj = objective(w)
                if obj > best_obj + 1e-6:
                    best_obj = obj
                    best_c = val
                    improved = True
            w[c] = best_c
        if not improved:
            break
    return w

## 3. 데이터·저장 예측 불러오기

In [4]:
train, test = load_data(DATA_DIR)   # test는 마지막 예측 파일에만 사용
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
classes = list(le.classes_)

dev_idx, hold_idx = verify()                     # 팀 고정 홀드아웃 (새로 만들지 않음)
y_dev, y_hold = y[dev_idx], y[hold_idx]
folds_dev = get_folds(y_dev, X=X.iloc[dev_idx].reset_index(drop=True), classes=le.classes_)

preds = load_selected_preds(CANDIDATE_KEYS, classes)
keys = list(preds)
oof_dev = [preds[k]["oof"][dev_idx] for k in keys]
oof_hold = [preds[k]["oof"][hold_idx] for k in keys]
print("모델", len(keys), "| 개발", len(dev_idx), "| 홀드아웃", len(hold_idx))

VERIFIED: dev=4960, holdout=1241; same IDs and row signatures
모델 4 | 개발 4960 | 홀드아웃 1241


## 4. 개발셋 교차 평가 — fold별 변화

In [5]:
# fold 4개로 가중치·승수를 정하고 남은 1개로 평가 × 5회 (평가 fold는 탐색에 쓰지 않음)
cv_base = np.zeros(len(y_dev), dtype=int)
cv_blend = np.zeros(len(y_dev), dtype=int)
cv_mult = np.zeros(len(y_dev), dtype=int)
fold_weights, fold_multipliers, fold_rows = [], [], []

for i, (tr, va) in enumerate(folds_dev):
    w = optimize_ensemble_weights([p[tr]  for p in oof_dev], y_dev[tr], n_trials=1500, seed=SEED + i)[0]
    tr_blend = sum(wi * p[tr] for wi, p in zip(w, oof_dev))
    va_blend = sum(wi * p[va] for wi, p in zip(w, oof_dev))
    mult = fit_class_multipliers_coord_ascent(tr_blend, y_dev[tr], n_iters=3, reg_lambda=0.003)
    fold_weights.append(w)
    fold_multipliers.append(mult)

    va_base = sum(p[va] for p in oof_dev[:3]) / 3
    cv_base[va], cv_blend[va], cv_mult[va] = va_base.argmax(1), va_blend.argmax(1), (va_blend * mult).argmax(1)
    f = lambda pred: f1_score(y_dev[va], pred[va], average="macro")
    fold_rows.append({"fold": i + 1, "기준": f(cv_base), "가중치": f(cv_blend) - f(cv_base),
                      "가중치+승수": f(cv_mult) - f(cv_base)})

fold_table = pd.DataFrame(fold_rows).round(4)
print(fold_table.to_string(index=False))
print("상승 fold 수 | 가중치:", (fold_table["가중치"] > 0).sum(), "/ 5 | 가중치+승수:", (fold_table["가중치+승수"] > 0).sum(), "/ 5")

 fold     기준     가중치  가중치+승수
    1 0.5276  0.0011 -0.0048
    2 0.5335 -0.0044 -0.0071
    3 0.5213  0.0144  0.0057
    4 0.5162  0.0062  0.0007
    5 0.5302  0.0035  0.0098
상승 fold 수 | 가중치: 4 / 5 | 가중치+승수: 3 / 5


## 5. 결과 — 개발 CV · 홀드아웃 1회

In [6]:
# 개발 CV(fold 밖 평가 모음) → 평균 가중치·승수로 홀드아웃 1회 평가
mf = lambda yt, pred: f1_score(yt, pred, average="macro")
base_dev, blend_dev, mult_dev = mf(y_dev, cv_base), mf(y_dev, cv_blend), mf(y_dev, cv_mult)

mean_w = np.mean(fold_weights, axis=0)
mean_w /= mean_w.sum()
mean_mult = np.mean(fold_multipliers, axis=0)

hold_base = sum(p for p in oof_hold[:3]) / 3
hold_blend = sum(wi * p for wi, p in zip(mean_w, oof_hold))
result = pd.DataFrame([
    {"방법": "기준 (3모델 균등)", "개발 CV": base_dev, "홀드아웃": mf(y_hold, hold_base.argmax(1))},
    {"방법": "4모델 가중치", "개발 CV": blend_dev, "홀드아웃": mf(y_hold, hold_blend.argmax(1))},
    {"방법": "4모델 가중치 + 클래스 승수", "개발 CV": mult_dev, "홀드아웃": mf(y_hold, (hold_blend * mean_mult).argmax(1))},
])
result["개발 CV 변화"] = result["개발 CV"] - base_dev
result["홀드아웃 변화"] = result["홀드아웃"] - result.loc[0, "홀드아웃"]
print(result.round(4).to_string(index=False))

print("\n최종 가중치:", {k.replace("A_model__", ""): round(float(wi), 4) for k, wi in zip(keys, mean_w)})
print("클래스 승수:", {c: round(float(m), 3) for c, m in zip(classes, mean_mult)})


              방법  개발 CV   홀드아웃  개발 CV 변화  홀드아웃 변화
     기준 (3모델 균등) 0.5291 0.5138    0.0000   0.0000
         4모델 가중치 0.5329 0.5229    0.0037   0.0091
4모델 가중치 + 클래스 승수 0.5306 0.5307    0.0015   0.0170

최종 가중치: {'12_weighted_xgb_E_cpu': 0.2525, '12_lr_position_bin50': 0.3587, 'lgbm_g3_hotspot3_bin_exact_1110': 0.2327, '12_g3_multihot_signature': 0.1561}
클래스 승수: {'ACC': 1.38, 'BLCA': 1.16, 'BRCA': 1.0, 'CESC': 1.15, 'COAD': 0.99, 'DLBC': 1.22, 'GBMLGG': 1.1, 'HNSC': 0.96, 'KIPAN': 1.07, 'KIRC': 1.08, 'LAML': 0.7, 'LGG': 0.67, 'LIHC': 1.0, 'LUAD': 1.1, 'LUSC': 0.96, 'OV': 0.86, 'PAAD': 1.05, 'PCPG': 0.8, 'PRAD': 0.95, 'SARC': 1.13, 'SKCM': 1.19, 'STES': 0.73, 'TGCT': 0.96, 'THCA': 1.05, 'THYM': 1.07, 'UCEC': 1.13}


## 6. 시험 제출 파일 (저장된 test 예측, submissions/)
- LB 시험용. 채택되면 최종 제출 파일은 submit/final_solution.ipynb(Colab CPU) 출력으로 다시 생성

In [7]:
# 저장된 test 예측으로 시험 제출 파일 생성 (submissions/, GitHub에 안 올라감)
test_blend = sum(wi * preds[k]["test"] for wi, k in zip(mean_w, keys))
make_submission(test, le.inverse_transform((test_blend * mean_mult).argmax(1)),
                "submission_04f_4model_weight_mult_1112.csv")

저장: /Users/admin/Documents/dacon_hackerton/4team_tuna/ah07-cancer-subtype/submissions/submission_04f_4model_weight_mult_1112.csv (2546행)


,ID,SUBCLASS
0,TEST_0000,STES
1,TEST_0001,STES
2,TEST_0002,THCA
3,TEST_0003,LGG
4,TEST_0004,HNSC
...,...,...
2541,TEST_2541,COAD
2542,TEST_2542,PAAD
2543,TEST_2543,UCEC
2544,TEST_2544,LAML


## 6-1. 안정성 확인 — 개발셋 분할(fold seed) 5개
- 이유: 같은 코드라도 개발셋 분할에 따라 결과가 +0.0015(Mac) ~ +0.0068(scikit-learn 1.6.1)로 달라짐
- 방법: 가중치·승수 탐색 설정은 그대로, get_folds의 seed만 5개로 바꿔 4번과 같은 교차 평가 반복
- 판단 기준 (실행 전 확정): 5개 평균 개발 CV 변화 +0.005 이상이면 제출 후보, 아니면 접기

In [8]:
# 판단 기준 (실행 전 확정): fold seed 5개 평균 개발 CV 변화가 +0.005 이상이면 제출 후보, 아니면 접기
STABILITY_SEEDS = [42, 7, 123, 2024, 777]
STABILITY_THRESHOLD = 0.005


def nested_dev_change(folds):
    """fold 4개로 가중치·승수를 정하고 남은 1개로 평가 → 개발 CV 변화 (가중치, 가중치+승수)."""
    pb, pw, pm = (np.zeros(len(y_dev), dtype=int) for _ in range(3))
    for i, (tr, va) in enumerate(folds):
        w = optimize_ensemble_weights([p[tr] for p in oof_dev], y_dev[tr], n_trials=1500, seed=SEED + i)[0]
        tr_blend = sum(wi * p[tr] for wi, p in zip(w, oof_dev))
        va_blend = sum(wi * p[va] for wi, p in zip(w, oof_dev))
        mult = fit_class_multipliers_coord_ascent(tr_blend, y_dev[tr], n_iters=3, reg_lambda=0.003)
        pb[va] = (sum(p[va] for p in oof_dev[:3]) / 3).argmax(1)
        pw[va], pm[va] = va_blend.argmax(1), (va_blend * mult).argmax(1)
    base = f1_score(y_dev, pb, average="macro")
    return f1_score(y_dev, pw, average="macro") - base, f1_score(y_dev, pm, average="macro") - base


rows = []
X_dev_raw = X.iloc[dev_idx].reset_index(drop=True)
for s in STABILITY_SEEDS:
    folds_s = get_folds(y_dev, X=X_dev_raw, classes=le.classes_, seed=s)   # 나누는 방법만 바꿈 (설정 그대로)
    dw, dm = nested_dev_change(folds_s)
    rows.append({"fold seed": s, "가중치": dw, "가중치+승수": dm})
    print(f"seed {s}: 가중치 {dw:+.4f} | 가중치+승수 {dm:+.4f}")

stab = pd.DataFrame(rows)
print("\n평균  가중치 {:+.4f} | 가중치+승수 {:+.4f}".format(stab["가중치"].mean(), stab["가중치+승수"].mean()))
print("상승 seed 수  가중치 {} / 5 | 가중치+승수 {} / 5".format((stab["가중치"] > 0).sum(), (stab["가중치+승수"] > 0).sum()))
best = stab[["가중치", "가중치+승수"]].mean().max()
print("판정:", "제출 후보" if best >= STABILITY_THRESHOLD else "기준 미달 → 접기", f"(기준 +{STABILITY_THRESHOLD})")

seed 42: 가중치 +0.0037 | 가중치+승수 +0.0015
seed 7: 가중치 +0.0042 | 가중치+승수 +0.0017
seed 123: 가중치 +0.0006 | 가중치+승수 -0.0010
seed 2024: 가중치 +0.0042 | 가중치+승수 +0.0053
seed 777: 가중치 +0.0022 | 가중치+승수 +0.0032

평균  가중치 +0.0030 | 가중치+승수 +0.0021
상승 seed 수  가중치 5 / 5 | 가중치+승수 4 / 5
판정: 기준 미달 → 접기 (기준 +0.005)


### 7. 결론 (Mac 실행 기준)
- 4모델 가중치 + 클래스 승수: 개발 CV +0.0015 (fold 3/5 상승), 홀드아웃 +0.0170
- 4모델 가중치만: 개발 CV +0.0037 (fold 4/5 상승), 홀드아웃 +0.0091
- 안정성 확인 (fold seed 5개 평균): 가중치 +0.0030 (5/5), 가중치 + 승수 +0.0021 (4/5) → 판단 기준(+0.005) 미달 → 제출 후보 아님
- 가중치만은 분할을 바꿔도 대부분 소폭 상승하지만 +0.005에 못 미침. 홀드아웃 상승은 개발 CV로 뒷받침되지 않음
- scikit-learn 1.6.1 환경에서도 같은 결론 (가중치 +0.0027, 가중치 + 승수 +0.0025)